# Análise exploratória — roubo a transeunte no Município do Rio de Janeiro

Notebook de exploração, prototipação e validação. A lógica reutilizável está em `src/`; aqui inspecionamos os dados passo a passo.

**Antes de rodar:** execute `python -m src.etl` na raiz do projeto.

In [ ]:
import sys, json
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
from IPython.display import display
from src import analise, visualizacao as viz

pd.options.display.float_format = '{:,.2f}'.format

## 1. Carga e estrutura

In [ ]:
mensal, anual = analise.carregar(RAIZ / 'data' / 'processed')
mensal.info()
display(mensal.head())
display(mensal.tail())

## 2. Qualidade e rastreabilidade
O relatório do ETL mostra qual arquivo foi usado e se houve divergências entre o que a fonte publica e o que recalculamos.

In [ ]:
qa = json.loads((RAIZ / 'data' / 'processed' / 'qa_report.json').read_text(encoding='utf-8'))
print('Arquivo:', qa['arquivo_bruto']); print('SHA-256:', qa['sha256_bruto']); print('Período:', qa['periodo'])
for k, v in qa['validacao'].items():
    print(f'{k}:', v)

In [ ]:
print('Nulos (mensal):'); display(mensal.isna().sum())
print('Anos parciais:'); display(anual.loc[~anual.ano_completo, ['ano', 'meses_validos']])

## 3. Estatística descritiva

In [ ]:
completos = anual[anual.ano_completo]
display(completos[['total_calculado', 'populacao', 'taxa_100k_calculada', 'variacao_pct']].describe())
cv = completos.total_calculado.std() / completos.total_calculado.mean()
print(f'Coeficiente de variação do total anual: {cv:.1%}')
print('Dispersão mensal dentro de cada ano (CV):')
display(mensal[~mensal.ano_parcial].groupby('ano').ocorrencias.agg(lambda s: s.std() / s.mean()).round(3).to_frame('cv_mensal').T)

### Correlações
Total e população crescem juntos? E a taxa, que já desconta a população?

In [ ]:
display(completos[['ano', 'total_calculado', 'populacao', 'taxa_100k_calculada']].corr().round(2))

## 4. Evolução anual (RQ1)

In [ ]:
display(anual[['ano', 'total_calculado', 'variacao_pct', 'populacao', 'taxa_100k_calculada']].round(1))
print(analise.tendencia_linear(completos.ano, completos.total_calculado))
viz.fig_evolucao_anual(anual)

## 5. Sazonalidade (RQ2)

In [ ]:
saz = analise.indice_sazonal(mensal)
saz['mes_nome'] = [m.capitalize() for m in analise.MESES_PT]
display(saz.round(3))
viz.fig_sazonalidade(saz)

In [ ]:
viz.fig_heatmap(mensal)

## 6. Pandemia (RQ3)
Razão entre cada mês e a média do mesmo mês em 2017–2019.

In [ ]:
cp = analise.comparacao_pandemia(mensal)
if cp is None:
    print('Período-base ausente nos dados.')
else:
    display(cp['razoes'].round(2))
    display(viz.fig_pandemia(cp))

## 7. Meses atípicos (outliers)
Regra de Tukey sobre o log da razão entre o valor e a mediana móvel centrada de 13 meses.

In [ ]:
display(analise.outliers_mensais(mensal).round(2))

## 8. Acumulado do último ano parcial

In [ ]:
ytd, corte = analise.comparacao_ytd(mensal)
print(f'Acumulado de janeiro ao mês {corte}:')
display(ytd.tail(8))

## 9. Conclusões preliminares
Escreva aqui, com suas palavras, o que os resultados acima respondem para RQ1, RQ2 e RQ3 (isso vira a base da seção 3 do artigo).